# Ingest: Clinical Trials Registry

Life sciences demo data. Pulls study records from the public
ClinicalTrials.gov API v2 (no auth, no API key) and persists them as a
managed Spark table — one row per registered trial.

Confirmed live field paths (2026-08-07):
`studies[].protocolSection.identificationModule.nctId`,
`...statusModule.overallStatus`, `...designModule.phases`,
`...conditionsModule.conditions`,
`...sponsorCollaboratorsModule.leadSponsor.name`,
`...designModule.enrollmentInfo.count`,
`...statusModule.startDateStruct.date`,
`...statusModule.completionDateStruct.date`.

Re-runnable: uses `CREATE DATABASE IF NOT EXISTS` and
`mode("overwrite")`, so re-running just pulls a fresh page of trials.

In [ ]:
import json
import urllib.request

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "clinical_trials"
TABLE_NAME = "trial_registry"
API_BASE_URL = "https://clinicaltrials.gov/api/v2/studies"
PAGE_SIZE = 500
MAX_PAGES = 4  # ~2000 trials -- enough to show real distribution across phase/status

## Fetch trial pages

Cursor-paginated via `nextPageToken`. Each study record is flattened down
to the fields this demo needs — the full API response is deeply nested
(`protocolSection.<module>.<field>`).

In [ ]:
def fetch_studies():
    rows = []
    page_token = None
    for _ in range(MAX_PAGES):
        url = f"{API_BASE_URL}?pageSize={PAGE_SIZE}"
        if page_token:
            url += f"&pageToken={page_token}"
        with urllib.request.urlopen(url) as resp:
            data = json.load(resp)

        for study in data.get("studies", []):
            protocol = study.get("protocolSection", {})
            identification = protocol.get("identificationModule", {})
            status = protocol.get("statusModule", {})
            design = protocol.get("designModule", {})
            conditions = protocol.get("conditionsModule", {}).get("conditions", [])
            sponsor = protocol.get("sponsorCollaboratorsModule", {}).get("leadSponsor", {})
            phases = design.get("phases", [])

            rows.append({
                "nct_id": identification.get("nctId"),
                "brief_title": identification.get("briefTitle"),
                "overall_status": status.get("overallStatus"),
                "phase": phases[0] if phases else "NA",
                "primary_condition": conditions[0] if conditions else None,
                "lead_sponsor": sponsor.get("name"),
                "enrollment_count": design.get("enrollmentInfo", {}).get("count"),
                "start_date": status.get("startDateStruct", {}).get("date"),
                "completion_date": status.get("completionDateStruct", {}).get("date"),
            })

        page_token = data.get("nextPageToken")
        if not page_token:
            break
    return rows

## Load dataset

In [ ]:
rows = fetch_studies()
df = spark.createDataFrame(rows)

## Create schema/database

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
COMMENT 'Clinical trial registry data used for life sciences demonstrations'
""")

## Persist as a managed Spark table

In [ ]:
df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TABLE_NAME}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_trials FROM {DATABASE_NAME}.{TABLE_NAME}").show()
spark.sql(f"DESCRIBE {DATABASE_NAME}.{TABLE_NAME}").show(truncate=False)